# Regional Data Inspection
Use this notebook to verify the `region` structure in CBS municipal price data.

Goal:
- Confirm whether municipality rows exist (commonly tagged as `(GM)`).
- Confirm province rows `(PV)` and national rows.
- Check if municipality names can be directly filtered from `region`.

In [3]:
import pandas as pd

from data.fetch_cbs import fetch_municipal_prices

prices = fetch_municipal_prices()
print("shape:", prices.shape)
print("columns:", prices.columns.tolist())
prices.head(100)

shape: (23095, 5)
columns: ['ID', 'region', 'period', 'avg_price', 'year']


,ID,region,period,avg_price,year
0,0,The Netherlands,1995,93750.0,1995
1,1,The Netherlands,1996,102607.0,1996
2,2,The Netherlands,1997,113163.0,1997
3,3,The Netherlands,1998,124540.0,1998
4,4,The Netherlands,1999,144778.0,1999
...,...,...,...,...,...
95,95,West-Nederland (LD),1997,116500.0,1997
96,96,West-Nederland (LD),1998,128373.0,1998
97,97,West-Nederland (LD),1999,150549.0,1999
98,98,West-Nederland (LD),2000,179105.0,2000


In [4]:
# Inspect how region labels are encoded, e.g. (GM), (PV), etc.
region_suffix = prices["region"].dropna().str.extract(r"\(([^)]+)\)$")[0]
region_suffix.value_counts(dropna=False).to_frame("count")

,count
0,
NaN,22196
PV,372
LD,124
L.,93
municipality,93
NH.,62
ZH.,62
Gld.,31
O.,31


In [5]:
# Compare likely municipalities vs provinces based on suffix.
municipalities = prices[prices["region"].str.endswith("(GM)", na=False)].copy()
provinces = prices[prices["region"].str.endswith("(PV)", na=False)].copy()

print("municipality rows:", len(municipalities))
print("province rows:", len(provinces))

print("\nmunicipality examples:")
print(municipalities["region"].drop_duplicates().sort_values().head(20).to_string(index=False))

print("\nprovince examples:")
print(provinces["region"].drop_duplicates().sort_values().head(20).to_string(index=False))

municipality rows: 0
province rows: 372

municipality examples:
Series([], )

province examples:
      Drenthe (PV)
    Flevoland (PV)
      Fryslân (PV)
   Gelderland (PV)
    Groningen (PV)
      Limburg (PV)
Noord-Brabant (PV)
Noord-Holland (PV)
   Overijssel (PV)
      Utrecht (PV)
      Zeeland (PV)
 Zuid-Holland (PV)


In [6]:
# Inspect region values without a suffix like (PV)/(LD).
# In this table, these are typically municipality names.
region_clean = prices["region"].dropna().str.strip()
no_suffix = region_clean[~region_clean.str.contains(r"\([^)]*\)$", regex=True)]

print("rows without suffix:", len(no_suffix))
print("unique without suffix:", no_suffix.nunique())
print("\nexamples without suffix:")
print("\n".join(sorted(no_suffix.drop_duplicates().head(40).tolist())))

rows without suffix: 22196
unique without suffix: 716

examples without suffix:
Aa en Hunze
Aalburg
Aalsmeer
Aalten
Aarle-Rixtel
Abcoude
Achtkarspelen
Akersloot
Alblasserdam
Albrandswaard
Alkemade
Alkmaar
Almelo
Almere
Alphen aan den Rijn
Alphen en Riel
Alphen-Chaam
Altena
Ambt Delden
Ambt Montfort
Ameland
Amerongen
Amersfoort
Ammerzoden
Amstelveen
Amsterdam
Andijk
Angerlo
Anloo
Anna Paulowna
Apeldoorn
Appingedam
Arcen en Velden
Arnemuiden
Arnhem
Assen
Asten
Avereest
Ter Aar
The Netherlands
